In [1]:
import os
import re
import sys
import json
import gzip
import copy
import math
import numpy
import random
import seaborn
import subprocess


import matplotlib
import matplotlib.pyplot as plt



# use default
seaborn.set_theme(style=None)
matplotlib.use('Agg')
matplotlib.rcParams.update(matplotlib.rcParamsDefault)


matplotlib.font_manager.fontManager.addfont('/wanglab/wzhang/.local/share/fonts/Helvetica.ttf')
matplotlib.rcParams['font.family'] = 'Helvetica'


samples = ["HG002", "HG005", "HG00438", "HG02257", "HG02486", "HG02622"]
phases = ["mat", "pat"]


sample_ratios = {
    "HG005":   0.835,

    "HG02622": 0.10,
    "HG002":   0.02,
    "HG02257": 0.02,
    "HG02486": 0.02,
    "HG00438": 0.005,
}


total_coverage = 4000





In [2]:
# Common functions



# Slow
def phred_to_numeric(qual_char):
    return ord(qual_char) - 33


phred_lookup_int = {chr(i): i - 33 for i in range(33, 255)}
def phred_to_int(qual_char):
    assert qual_char in phred_lookup_int
    return phred_lookup_int[qual_char]


phred_lookup_prob = {char: 10 ** (-(phred_score / 10.0)) for char, phred_score in phred_lookup_int.items()}
def phred_to_prob(qual_char):
    assert qual_char in phred_lookup_prob
    return phred_lookup_prob[qual_char]


def prob_to_phred(prob):
    return -10 * math.log10(prob)

def prob_to_phred_int(prob):
    return int(round(prob_to_phred(prob), 0))

def prob_to_phred_char(prob):
    return chr(prob_to_phred_int(prob) + 33)


# Just to test
for i in range(33, 255):
    pass
    p = phred_to_prob(chr(i))

    assert phred_to_int(chr(i)) == prob_to_phred_int(p)
    assert chr(i) == prob_to_phred_char(p)

    # print(chr(i), phred_to_int(chr(i)), p)







In [3]:
# Final QC COMMON


fig_folder = f"/scratch/wzhang/projects/SMaHT/bam_stat/fig2/"

target_cov = {
    "HG002": 40,
    "HG005": 1670,
    "HG00438": 10,
    "HG02257": 40,
    "HG02486": 40,
    "HG02622": 200,

}


colors = {
    "R100X2": "#1f77b4",
    "SIM100": "#ff7f0e",
    "SIM4000": "#2ca02c",
}

fig_size = (8, 4.34)


In [6]:
# Final QC PART 1

figures = {}


# fig2: sequence length distribution
fig2, axes2 = plt.subplots(1, 1, figsize=fig_size, sharex=True)

# fig3: number of passes distribution
fig3, axes3 = plt.subplots(1, 1, figsize=fig_size, sharex=True)

# fig4: CG frequency distribution
fig4, axes4 = plt.subplots(1, 1, figsize=fig_size, sharex=True)



# figures["1"] = [fig1, axes1]
figures["1"] = [fig2, axes2]
figures["2"] = [fig3, axes3]
figures["3"] = [fig4, axes4]






def contig_stat_reader(fp):
    contig_stat = {}

    with open(fp) as fh:
        for l in fh:
            l = l.strip().split()
            ci, cname, cl = l
            ci = int(ci)
            cl = int(cl)
            contig_stat[ci] = cl

        return contig_stat




si = -1
for sample in ["R100X2", "SIM100", "SIM4000"]:
    si += 1

    print(sample)

    line_style = "solid"
    if sample == "SIM4000":
        line_style = "dashed"


    label = sample
    if sample == "SIM100":
        label = "Sim 100x"
    elif sample == "SIM4000":
        label = "Sim 4000x"
    elif sample == "R100X2":
        label = "Revio 100X"



    fp = f"/scratch/wzhang/projects/SMaHT/bam_stat/step2/{sample}.json"
    if sample == "SIM4000":
        # DEBUGGING ONLY
        pass
        # fp = f"/scratch/wzhang/projects/SMaHT/bam_stat/step2/SIM100.json"
        # fp = f"/scratch/wzhang/projects/SMaHT/bam_stat/step2/HG00438.json"

    res = json.load(open(fp))
    read_lengths, pass_nums, cg_freqs, cov_by_seq = res

    bases = sum(read_lengths)
    ave_len = bases / len(read_lengths)

    print(f"Total bases: {bases/1e9:.1f}G")
    print(f"Average read length: {ave_len:.0f}")
    print(f"Max pass num: {max(pass_nums)}")


    ave_len_y0 = 0.000105
    if sample.startswith("SIM"):
        ave_len_y0 = 0.00008

    ave_len_x = [ave_len, ave_len]
    ave_len_y = [0, ave_len_y0]



    # seaborn.kdeplot(read_lengths, ax=axes2, label=label, fill=False, color=colors[sample], linestyle=line_style)
    axes2.hist(read_lengths, bins=100, alpha=0.3, range=(0, 50000), weights=numpy.ones_like(read_lengths)/len(read_lengths),  color=colors[sample], label=label)
    # Show average
    # axes2.plot(ave_len_x, ave_len_y, color=colors[sample], linestyle=line_style)
    axes2.legend()
    axes2.title.set_text("Read Length Distribution")
    axes2.set_xlabel("Read Length")
    axes2.set_ylabel("Frequency")

    pass_num_x = list(range(1, 61))
    pass_num_y = [pass_nums.count(p)/len(pass_nums) for p in pass_num_x]
    axes3.plot(pass_num_x, pass_num_y, label=label, linestyle=line_style, color=colors[sample])
    axes3.legend()
    axes3.title.set_text("Number of Passes")
    axes3.set_xlabel("Pass #")
    axes3.set_ylabel("Frequency")

    # smooth kde
    # seaborn.kdeplot(cg_freqs, ax=axes4, label=label, fill=False, bw_method=0.2, linestyle=line_style, color=colors[sample])
    axes4.hist(cg_freqs, bins=70, alpha=0.3, range=(0.1, 0.8), weights=numpy.ones_like(cg_freqs)/len(cg_freqs),  color=colors[sample], label=label)
    axes4.legend()
    axes4.title.set_text("GC Content Percentage")
    axes4.set_xlabel("GC Content Percentage")
    axes4.set_ylabel("Frequency")




for fname, figs in figures.items():
    fig, axes = figs
    fig_fp = f"{fig_folder}/{fname}.pdf"

    fig.savefig(fig_fp, bbox_inches='tight')










R100X2
Total bases: 308.1G
Average read length: 22152
Max pass num: 60
SIM100
Total bases: 289.8G
Average read length: 21981
Max pass num: 7
SIM4000
Total bases: 11592.8G
Average read length: 21982
Max pass num: 7


In [ ]:
# Final QC PART 2: Per-read Average PHRED Score Distribution



figures = {}

phred_line_limit = 1e12



fig_size = (8, 4.34)
fig1, axes1 = plt.subplots(1, 1, figsize=fig_size)

figures["5"] = [fig1, axes1]









si = -1
for sample in ["R100X2", "SIM100", "SIM4000"]:
    si += 1

    print(sample)

    line_style = "solid"
    if sample == "SIM4000":
        line_style = "dashed"


    label = sample
    if sample == "SIM100":
        label = "Sim 100x"
    elif sample == "SIM4000":
        label = "Sim 4000x"
    elif sample == "R100X2":
        label = "Revio 100X"



    fp1 = f"/scratch/wzhang/projects/SMaHT/bam_stat/step1/{sample}.txt"
    read_phred_scores = []
    with open(fp1) as fh:
        for l in fh:
            rn, seql, atcg_freq, phred_freq, pn = l.strip().split("\t")
            seql = int(seql)
            phred_freq = eval(phred_freq)

            assert seql == sum(phred_freq.values())
            prob_total = 0
            for phred, freq in phred_freq.items():
                prob_total += phred_to_prob(phred) * freq

            prob_ave = prob_total / seql
            phred_ave = prob_to_phred(prob_ave)
            read_phred_scores.append(phred_ave)

            if len(read_phred_scores) > phred_line_limit:
                break

    # seaborn.kdeplot(read_phred_scores, ax=axes1, label=label, fill=False, color=colors[sample], linestyle=line_style, bw_method=1)
    axes1.hist(read_phred_scores, bins=100, alpha=0.3, range=(0, 101), color=colors[sample], weights=numpy.ones_like(read_phred_scores)/len(read_phred_scores), label=label)
    # Show average
    # axes2.plot(ave_len_x, ave_len_y, color=colors[sample], linestyle=line_style)
    axes1.legend()
    axes1.title.set_text("Per-read Average PHRED Score Distribution")
    axes1.set_xlabel("PHRED score")
    axes1.set_ylabel("Frequency")







save = True
for fname, figs in figures.items():
    fig, axes = figs
    fig_fp = f"{fig_folder}/{fname}.pdf"
    save = False

    fig.savefig(fig_fp, bbox_inches='tight')











R100X2
SIM100
SIM4000


In [4]:
# Final QC PART 3: Post-alignment MAPQ and Alignment Percentage Distribution


data = {}

def alignment_report_reader(fp):
    mapq_d = []
    align_len_per_d = []
    with open(fp) as fh:
        for l in fh:
            l = l.strip().split("\t")
            seq_len = int(l[1])
            mapq = int(l[2])
            map_len = int(l[3])
            map_per = map_len / seq_len *100

            mapq_d.append(mapq)
            align_len_per_d.append(map_per)

            if len(mapq_d) > 1000000:
                pass

    return mapq_d, align_len_per_d


for sample in ["R100X2", "SIM100", "SIM4000"]:

    sx = sample
    if sample.startswith("SIM"):
        sx += "x"
    else:
        sx = sample[:-2] + "x"

    report_fp = f"/clearwater/wzhang/projects/SMaHT/bam_stat/step3/{sx}.report.txt"
    mapq_d, align_len_per_d = alignment_report_reader(report_fp)
    data[sample] = (mapq_d, align_len_per_d)



fig_size = (8, 4.34)
fig1, ax1 = plt.subplots(1, 1, figsize=fig_size)
fig2, ax2 = plt.subplots(1, 1, figsize=fig_size)
for sample in data:
    mapq_d, align_len_per_d = data[sample]
    ls = "solid"
    if sample == "SIM4000":
        ls = "dashed"

    if sample == "SIM100":
        label = "Sim 100x"
    elif sample == "SIM4000":
        label = "Sim 4000x"
    elif sample == "R100X2":
        label = "Revio 100X"

    # seaborn.kdeplot(mapq_d, ax=ax1, label=f"{label}", color=colors[sample], linestyle=ls)
    # seaborn.kdeplot(align_len_per_d, ax=ax2, label=f"{label}", color=colors[sample], linestyle=ls)
    ax1.hist(mapq_d, bins=100, alpha=0.3, range=(0, 61), color=colors[sample], weights=numpy.ones_like(mapq_d)/len(mapq_d), label=sample)



for ax in [ax1, ax2]:
    ax.legend()

ax1.set_title("MAPQ Distribution")
ax2.set_title("Alignment Percentage Distribution")

ax1.set_xlabel("MAPQ")
ax2.set_xlabel("Alignment Percentage")

ax1.set_ylabel("Frequency")
ax2.set_ylabel("Frequency")


fig1.savefig("/clearwater/wzhang/projects/SMaHT/bam_stat/step3/mapq.pdf", bbox_inches='tight')
fig2.savefig("/clearwater/wzhang/projects/SMaHT/bam_stat/step3/alignment_percentage.pdf", bbox_inches='tight')



/tmp/ipykernel_1750079/2880097494.py:60: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend()


In [5]:
# Final QC PART 4: coverage Distribution

data = {}

def coverage_reader(fp, cap=4000):
    res = []
    with gzip.open(fp, "rt") as fh:
        for l in fh:
            l = l.strip().split("\t")
            a,b,c,d = l
            d= float(d)
            if d > 2*cap:
                d = 2*cap
                continue
            res.append(d)

            if len(res) > 100000:
                pass
                #break
    return res



for sample in ["R100X2", "SIM100", "SIM4000"]:
    sx = sample
    if sample.startswith("SIM"):
        sx += "x"
    else:
        sx = sample[:-2] + "x"

    cap = 100
    if sample == "SIM4000":
        cap = 4000


    report_fp = f"/scratch/wzhang/projects/SMaHT/alignment_coverage/{sx}/sample.depth.100.tsv.gz"
    data[sample] = coverage_reader(report_fp, cap=cap)



fig_size = (8, 4.34)

fig1, ax1 = plt.subplots(1, 1, figsize=fig_size)
fig2, ax2 = plt.subplots(1, 1, figsize=fig_size)
fig3, ax3 = plt.subplots(1, 1, figsize=fig_size)

axes3 = [ax1, ax2, ax3]

plt.subplots_adjust(hspace=0.3)
for sample in data:
    ax = axes3[list(data.keys()).index(sample)]
    cov_d = data[sample]


    ls = "solid"
    if sample == "SIM4000":
        ls = "dashed"

    if sample == "SIM100":
        label = "Sim 100x"

    elif sample == "SIM4000":
        label = "Sim 4000x"
    elif sample == "R100X2":
        label = "Revio 100X"

    # seaborn.kdeplot(cov_d, ax=ax, label=f"{label}", color=colors[sample], linestyle=ls)
    cap = 100
    if sample == "SIM4000":
        cap = 4000
    ax.hist(cov_d, bins=100, range=(0, 2*cap), color=colors[sample], weights=numpy.ones_like(cov_d)/len(cov_d), label=sample)


for ax in axes3:
    ax.legend()
    ax.set_title("100bp Bin Coverage Distribution")
    ax.set_xlabel("Coverage")
    ax.set_ylabel("Frequency")


for fi, fig in enumerate([fig1, fig2, fig3]):
    fig.savefig(f"/clearwater/wzhang/projects/SMaHT/bam_stat/step3/coverage_{fi}.pdf", bbox_inches='tight')

